In [1]:
import pandas as pd

from src.config.paths import ProjectPaths
from src.data.target import TargetBuilder

In [2]:
df = pd.read_parquet(ProjectPaths.EDA_DATASET)

print(f"Input shape: {df.shape}")

Input shape: (2260701, 94)


In [3]:
target_builder = TargetBuilder()

model_df = target_builder.build(df)

print(f"Modeling population shape: {model_df.shape}")

2026-08-10 20:12:56 | INFO     | src.data.target | Building observed-outcome target.
2026-08-10 20:12:58 | INFO     | src.data.target | Target construction completed. Retained 1348099 of 2260701 rows.


Modeling population shape: (1348099, 95)


In [4]:
display(model_df["loan_status"].value_counts().to_frame("count"))

,count
loan_status,
Fully Paid,1076751
Charged Off,268559
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,40


In [5]:
display(
    model_df["default"]
    .value_counts()
    .rename(index={0: "Non-default", 1: "Default"})
    .to_frame("count")
)

,count
default,
Non-default,1078739
Default,269360


In [6]:
population_summary = pd.DataFrame(
    {
        "metric": [
            "EDA input rows",
            "Observed-outcome rows",
            "Excluded rows",
            "Observed-outcome percentage",
            "Default rate",
        ],
        "value": [
            len(df),
            len(model_df),
            len(df) - len(model_df),
            round(len(model_df) / len(df) * 100, 2),
            round(model_df["default"].mean() * 100, 2),
        ],
    }
)

display(population_summary)

,metric,value
0,EDA input rows,2260701.00
1,Observed-outcome rows,1348099.00
2,Excluded rows,912602.00
3,Observed-outcome percentage,59.63
4,Default rate,19.98


In [7]:
review_features = [
    "funded_amnt",
    "funded_amnt_inv",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "issue_d",
    "initial_list_status",
]

review_summary = pd.DataFrame(
    {
        "feature": review_features,
        "dtype": [model_df[col].dtype for col in review_features],
        "missing_pct": [model_df[col].isna().mean() * 100 for col in review_features],
        "n_unique": [model_df[col].nunique(dropna=True) for col in review_features],
    }
)

display(review_summary.round(2))

,feature,dtype,missing_pct,n_unique
0,funded_amnt,float64,0.0,1560
1,funded_amnt_inv,float64,0.0,10041
2,int_rate,float64,0.0,672
3,installment,float64,0.0,83531
4,grade,object,0.0,7
5,sub_grade,object,0.0,35
6,issue_d,object,0.0,139
7,initial_list_status,object,0.0,2


In [8]:
for col in review_features:
    print(f"\n{'=' * 60}")
    print(col)
    print(model_df[col].dropna().head(10).tolist())


funded_amnt
[3600.0, 24700.0, 20000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0, 1400.0]

funded_amnt_inv
[3600.0, 24700.0, 20000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0, 1400.0]

int_rate
[13.99, 11.99, 10.78, 22.45, 13.44, 9.17, 8.49, 6.49, 11.48, 12.88]

installment
[123.03, 820.28, 432.66, 289.91, 405.18, 637.58, 631.26, 306.45, 263.74, 47.1]

grade
['C', 'C', 'B', 'F', 'C', 'B', 'B', 'A', 'B', 'C']

sub_grade
['C4', 'C1', 'B4', 'F1', 'C3', 'B2', 'B1', 'A2', 'B5', 'C2']

issue_d
['Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015']

initial_list_status
['w', 'w', 'w', 'w', 'w', 'f', 'w', 'w', 'w', 'w']


In [9]:
from src.data.leakage import LeakageAnalyzer

leakage_report = LeakageAnalyzer().analyze(model_df)

approved_features = leakage_report.keep

feature_types = pd.DataFrame(
    {
        "feature": approved_features,
        "dtype": [model_df[col].dtype for col in approved_features],
        "n_unique": [model_df[col].nunique(dropna=True) for col in approved_features],
        "missing_pct": [model_df[col].isna().mean() * 100 for col in approved_features],
    }
)

display(
    feature_types.sort_values(
        ["dtype", "n_unique"],
        ascending=[True, False],
    )
)

2026-08-10 20:12:59 | INFO     | src.data.leakage | Loading leakage rules from D:\Prep Stuff\Projects\credit-risk-platform\configs\leakage_rules.yaml
2026-08-10 20:12:59 | WARNING  | src.data.leakage | No leakage rule configured for 'default'.
2026-08-10 20:12:59 | INFO     | src.data.leakage | Leakage analysis completed. KEEP=85 REVIEW=8 DROP=0 LEAKAGE=0 TARGET=1 UNKNOWN=1


,feature,dtype,n_unique,missing_pct
81,tot_hi_cred_lim,float64,427447,5.212970
32,tot_cur_bal,float64,400464,5.212970
82,total_bal_ex_mort,float64,177997,3.711152
84,total_il_high_credit_limit,float64,162550,5.212970
38,total_bal_il,float64,118365,60.118953
...,...,...,...,...
4,home_ownership,object,6,0.000000
6,verification_status,object,3,0.000000
29,verification_status_joint,object,3,98.100955
1,term,object,2,0.000000


In [10]:
pd.set_option("display.max_rows", 100)

display(
    feature_types.sort_values(
        ["dtype", "n_unique"],
        ascending=[True, False],
    )
)

,feature,dtype,n_unique,missing_pct
81,tot_hi_cred_lim,float64,427447,5.212970
32,tot_cur_bal,float64,400464,5.212970
82,total_bal_ex_mort,float64,177997,3.711152
84,total_il_high_credit_limit,float64,162550,5.212970
38,total_bal_il,float64,118365,60.118953
21,revol_bal,float64,84084,0.000000
49,avg_cur_bal,float64,76864,5.214602
50,bc_open_to_buy,float64,74924,4.739563
5,annual_inc,float64,64462,0.000297
42,max_bal_bc,float64,27510,60.118953


# Phase 5 — Data Preprocessing

## Objective

This notebook documents and validates the preprocessing decisions for the credit-risk modeling dataset.

The notebook is used for analysis and validation only. Production preprocessing logic is implemented under `src/features/` and `src/pipelines/` so that the pipeline does not depend on notebook execution.

The preprocessing stage converts the EDA-approved modeling population into a consistent feature representation suitable for downstream feature engineering and model development.

---

## 5.1 Modeling Population

The modeling population consists only of loans whose eventual outcomes are observed.

The target is constructed using `TargetBuilder` from `src/data/target.py`.

Loans with ongoing outcomes are excluded because their eventual default status cannot yet be observed reliably.

### Population Summary

| Metric                      |     Value |
| --------------------------- | --------: |
| Input EDA population        | 2,260,701 |
| Observed-outcome population | 1,348,099 |
| Excluded loans              |   912,602 |
| Retained population         |    59.63% |
| Default rate                |    19.98% |

The resulting `default` variable is the binary modeling target:

* `0` → Non-default
* `1` → Default

---

## 5.2 Feature Eligibility

Feature eligibility was established during Data Understanding and EDA.

| Category                        |  Count |
| ------------------------------- | -----: |
| Approved modeling features      |     85 |
| Dropped features                |      8 |
| Original target (`loan_status`) |      1 |
| Derived target (`default`)      |      1 |
| **Total columns**               | **95** |

The eight reviewed features were excluded based on the selected prediction point:

> **Prediction point: underwriting/approval, before loan origination and finalization of loan terms.**

Therefore, features that represent finalized loan terms, funding decisions, or lender-assigned underwriting outcomes are not used as model inputs.

### Excluded Features

* `funded_amnt`
* `funded_amnt_inv`
* `int_rate`
* `installment`
* `grade`
* `sub_grade`
* `issue_d`
* `initial_list_status`

This decision is based on prediction-point eligibility rather than their statistical relationship with the target.

For example, `int_rate` demonstrated a strong monotonic relationship with default during EDA. However, predictive strength alone does not make a feature valid: the feature must also be available at the defined prediction point.

---

## 5.3 Preprocessing Principles

The preprocessing design follows these principles:

1. **Do not use the target as an input feature.**
2. **Do not fit preprocessing statistics on the full dataset.**
3. All learned preprocessing parameters will be fitted using the training population only.
4. The same fitted transformations must be reusable on validation, test, and future inference data.
5. Missingness is not automatically treated as ordinary missing data; its meaning depends on the feature.
6. Categorical variables must safely handle previously unseen categories.
7. High-cardinality categorical variables must not be blindly one-hot encoded.
8. Feature engineering is kept separate from basic preprocessing.
9. Model-specific transformations such as scaling will be handled according to the downstream model requirements.

---

## 5.4 Preprocessing Treatment Strategy

Features are grouped according to their semantic behavior rather than only their pandas datatype.

| Feature group                | Treatment                                      |
| ---------------------------- | ---------------------------------------------- |
| Standard numerical           | Median imputation                              |
| Event-history numerical      | Missingness indicator + specialized imputation |
| Structural numerical         | Missingness indicator + imputation             |
| Joint numerical              | Missingness indicator + imputation             |
| Joint categorical            | Explicit missing category                      |
| Regular categorical          | Explicit missing category + one-hot encoding   |
| High-cardinality categorical | Deferred for Feature Engineering               |
| Datetime                     | Datetime normalization                         |
| Dropped features             | Excluded from feature matrix                   |
| Target                       | Kept separately from model inputs              |

---

## 5.5 Preprocessing vs Feature Engineering

Preprocessing makes existing features usable without changing the underlying business meaning of the variables.

Feature engineering creates new representations or variables from existing information.

Examples that belong to later Feature Engineering include:

* credit-history duration from `earliest_cr_line`
* FICO-derived representations
* income-to-loan ratios
* installment-to-income measures
* utilization aggregates
* domain-specific interaction features

These transformations are intentionally not implemented in this phase.


In [11]:
import pandas as pd

from src.features.transformers import MissingIndicatorImputer

test_df = pd.DataFrame(
    {
        "income": [50000, 60000, None, 80000],
        "dti": [10.0, None, 20.0, 30.0],
    }
)

transformer = MissingIndicatorImputer()

result = transformer.fit_transform(test_df)

display(result)

,income,income__missing,dti,dti__missing
0,50000.0,0,10.0,0
1,60000.0,0,20.0,1
2,60000.0,1,20.0,0
3,80000.0,0,30.0,0


In [12]:
import pandas as pd

from src.features.transformers import EventHistoryImputer

test_df = pd.DataFrame(
    {
        "mths_since_last_delinq": [12.0, 24.0, None, 36.0],
        "mths_since_last_record": [10.0, None, 20.0, 30.0],
    }
)

transformer = EventHistoryImputer()

result = transformer.fit_transform(test_df)

display(result)

,mths_since_last_delinq,mths_since_last_delinq__missing,mths_since_last_record,mths_since_last_record__missing
0,12.0,0,10.0,0
1,24.0,0,31.0,1
2,37.0,1,20.0,0
3,36.0,0,30.0,0


In [13]:
import pandas as pd

from src.features.transformers import StructuralMissingnessImputer

test_df = pd.DataFrame(
    {
        "open_acc_6m": [2.0, None, None, 4.0],
        "open_act_il": [1.0, None, None, 3.0],
        "inq_last_12m": [3.0, None, 2.0, 4.0],
    }
)

transformer = StructuralMissingnessImputer()

result = transformer.fit_transform(test_df)

display(result)

,open_acc_6m,open_acc_6m__missing,open_act_il,open_act_il__missing,inq_last_12m,inq_last_12m__missing,structural_block__missing
0,2.0,0,1.0,0,3.0,0,0
1,3.0,1,2.0,1,3.0,1,1
2,3.0,1,2.0,1,2.0,0,0
3,4.0,0,3.0,0,4.0,0,0


In [14]:
from src.features.preprocessing import (
    CATEGORICAL_FEATURES,
    EVENT_HISTORY_FEATURES,
    JOINT_CATEGORICAL_FEATURES,
    JOINT_NUMERICAL_FEATURES,
    STANDARD_NUMERICAL_FEATURES,
    STRUCTURAL_NUMERICAL_FEATURES,
)

preprocessing_groups = {
    "standard_numerical": STANDARD_NUMERICAL_FEATURES,
    "event_history": EVENT_HISTORY_FEATURES,
    "structural": STRUCTURAL_NUMERICAL_FEATURES,
    "joint_numerical": JOINT_NUMERICAL_FEATURES,
    "categorical": CATEGORICAL_FEATURES + JOINT_CATEGORICAL_FEATURES,
}

routed_features = [
    feature for features in preprocessing_groups.values() for feature in features
]

print("Total routed features:", len(routed_features))
print("Unique routed features:", len(set(routed_features)))

duplicates = [
    feature for feature in set(routed_features) if routed_features.count(feature) > 1
]

print("Duplicated features:", duplicates)

Total routed features: 81
Unique routed features: 81
Duplicated features: []


## 5.6 Train / Validation / Test Split

The observed-outcome modeling population is divided into training,
validation, and test populations using a 70/15/15 split.

The split is stratified on the binary `default` target so that the default
rate remains approximately consistent across all populations.

The preprocessing pipeline is fitted exclusively on the training data.
Validation and test data are transformed using the parameters learned from
the training population.

This prevents information from the validation or test populations from
influencing imputation or other learned preprocessing parameters.

In [15]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["loan_status", "default"])
y = model_df["default"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

Train: (943669, 93)
Validation: (202215, 93)
Test: (202215, 93)


In [16]:
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [
            len(y_train),
            len(y_valid),
            len(y_test),
        ],
        "default_rate": [
            y_train.mean() * 100,
            y_valid.mean() * 100,
            y_test.mean() * 100,
        ],
    }
)

display(split_summary.round(2))

,split,rows,default_rate
0,train,943669,19.98
1,validation,202215,19.98
2,test,202215,19.98


In [17]:
from src.features.preprocessing import build_preprocessor

preprocessor = build_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_valid_processed.shape)
print("Test:", X_test_processed.shape)

Train: (943669, 241)
Validation: (202215, 241)
Test: (202215, 241)


In [18]:
feature_names = preprocessor.get_feature_names_out()

print("Processed feature count:", len(feature_names))

for i, feature in enumerate(feature_names):
    print(f"{i:03d} | {feature}")

Processed feature count: 241
000 | standard_numerical__loan_amnt
001 | standard_numerical__loan_amnt__missing
002 | standard_numerical__annual_inc
003 | standard_numerical__annual_inc__missing
004 | standard_numerical__dti
005 | standard_numerical__dti__missing
006 | standard_numerical__delinq_2yrs
007 | standard_numerical__delinq_2yrs__missing
008 | standard_numerical__fico_range_low
009 | standard_numerical__fico_range_low__missing
010 | standard_numerical__fico_range_high
011 | standard_numerical__fico_range_high__missing
012 | standard_numerical__inq_last_6mths
013 | standard_numerical__inq_last_6mths__missing
014 | standard_numerical__mths_since_recent_bc
015 | standard_numerical__mths_since_recent_bc__missing
016 | standard_numerical__open_acc
017 | standard_numerical__open_acc__missing
018 | standard_numerical__pub_rec
019 | standard_numerical__pub_rec__missing
020 | standard_numerical__revol_bal
021 | standard_numerical__revol_bal__missing
022 | standard_numerical__revol_util
0

In [19]:
feature_names = preprocessor.get_feature_names_out()

print("Processed feature count:", len(feature_names))
print(feature_names[:20])

Processed feature count: 241
['standard_numerical__loan_amnt' 'standard_numerical__loan_amnt__missing'
 'standard_numerical__annual_inc'
 'standard_numerical__annual_inc__missing' 'standard_numerical__dti'
 'standard_numerical__dti__missing' 'standard_numerical__delinq_2yrs'
 'standard_numerical__delinq_2yrs__missing'
 'standard_numerical__fico_range_low'
 'standard_numerical__fico_range_low__missing'
 'standard_numerical__fico_range_high'
 'standard_numerical__fico_range_high__missing'
 'standard_numerical__inq_last_6mths'
 'standard_numerical__inq_last_6mths__missing'
 'standard_numerical__mths_since_recent_bc'
 'standard_numerical__mths_since_recent_bc__missing'
 'standard_numerical__open_acc' 'standard_numerical__open_acc__missing'
 'standard_numerical__pub_rec' 'standard_numerical__pub_rec__missing']


In [20]:
for name, transformer, columns in preprocessor.transformers_:
    print(f"{name}: {len(columns)} input features")

standard_numerical: 53 input features
event_history: 7 input features
structural: 11 input features
joint_numerical: 2 input features
categorical: 8 input features
remainder: 12 input features


In [21]:
from src.features.registry import HIGH_CARDINALITY_FEATURES

print("\nDeferred features:")
print("High-cardinality:", HIGH_CARDINALITY_FEATURES)
print("Datetime:", ("earliest_cr_line",))


Deferred features:
High-cardinality: ('emp_title', 'title', 'zip_code')
Datetime: ('earliest_cr_line',)


In [22]:
for name, transformer, columns in preprocessor.transformers_:
    print(f"\n{name}:")
    print(f"  Input features: {len(columns)}")
    print(f"  Features: {list(columns)}")


standard_numerical:
  Input features: 53
  Features: ['loan_amnt', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_recent_bc', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_open_past_24mths', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'total_rev_hi_lim', 'il_util', 'all_util']

even

In [23]:
from src.features.preprocessing import prepare_preprocessed_data

artifacts = prepare_preprocessed_data()

2026-08-10 20:13:28 | INFO     | src.data.loader | Loading dataset: accepted_2007_to_2018Q4.csv
2026-08-10 20:14:10 | INFO     | src.data.target | Building observed-outcome target.
2026-08-10 20:14:11 | INFO     | src.data.target | Target construction completed. Retained 1348099 of 2260701 rows.


In [25]:
import pandas as pd

from src.config.paths import ProjectPaths

for name, path in {
    "train": ProjectPaths.TRAIN_PROCESSED,
    "validation": ProjectPaths.VALIDATION_PROCESSED,
    "test": ProjectPaths.TEST_PROCESSED,
}.items():
    df = pd.read_parquet(path)

    print(
        f"{name}: "
        f"shape={df.shape}, "
        f"default_rate={df['default'].mean():.4f}, "
        f"missing={df.isna().sum().sum()}"
    )

train: shape=(943669, 242), default_rate=0.1998, missing=0
validation: shape=(202215, 242), default_rate=0.1998, missing=0
test: shape=(202215, 242), default_rate=0.1998, missing=0
